In [ ]:
#!/usr/bin/env python3
"""
Lane Count Extraction Script (TomTom Edition)

Reads:  video_name, frame_number, lat, lon
Writes: video_name, frame_number, lat, lon, lane_count

For each coordinate, builds a short local route using TomTom Routing API and
parses lane guidance to estimate the number of lanes near that location.

Env:
    TOMTOM_API_KEY  (or hardcode in TOMTOM_API_KEY below)
"""

import os
import csv
import time
import math
import requests

TOMTOM_API_KEY = os.getenv("TOMTOM_API_KEY", "YOUR_TOMTOM_API_KEY")
TOMTOM_ROUTE_URL = "https://api.tomtom.com/routing/1/calculateRoute/{start}:{end}/json"

# --- small utilities ---------------------------------------------------------

def offset_point(lat, lon, meters_forward=40.0, bearing_deg=0.0):
    """
    Returns a new lat, lon that is meters_forward away from (lat, lon) along bearing_deg.
    If you don't have heading per frame, we can just push 'north' (bearing=0).
    """
    R = 6371000.0  # earth radius in meters
    bearing = math.radians(bearing_deg)
    lat1 = math.radians(lat)
    lon1 = math.radians(lon)
    dR = meters_forward / R

    lat2 = math.asin(math.sin(lat1) * math.cos(dR) + math.cos(lat1) * math.sin(dR) * math.cos(bearing))
    lon2 = lon1 + math.atan2(math.sin(bearing) * math.sin(dR) * math.cos(lat1),
                             math.cos(dR) - math.sin(lat1) * math.sin(lat2))
    return (math.degrees(lat2), math.degrees(lon2))

def extract_lane_count_from_tomtom(resp_json):
    """
    Heuristic: check guidance->instructions for lane info,
    or sections of type 'Lanes'. Return max lanes seen in nearby instructions.
    """
    lane_counts = []

    # 1) Guidance instructions (if present)
    guidance = resp_json.get("guidance", {})
    for instr in guidance.get("instructions", []) or []:
        # Some responses include 'lanes' arrays in instructions
        # Example: instr.get("lanes") -> list of lane descriptors
        if "lanes" in instr and isinstance(instr["lanes"], list) and instr["lanes"]:
            lane_counts.append(len(instr["lanes"]))

        # Some responses include 'laneGuidance' objects
        lg = instr.get("laneGuidance")
        if lg and isinstance(lg, dict):
            lanes = lg.get("lanes")
            if isinstance(lanes, list) and lanes:
                lane_counts.append(len(lanes))

    # 2) Sections of type 'Lanes' (if present)
    for section in resp_json.get("sections", []) or []:
        if section.get("sectionType", "").lower() == "lanes":
            lanes = section.get("lanes")
            if isinstance(lanes, list) and lanes:
                lane_counts.append(len(lanes))

    if lane_counts:
        # choose the mode or simply the max seen (more conservative for multi-lane roads)
        return max(lane_counts)
    return None

def get_lane_count_tomtom(lat, lon, meters_ahead=40, bearing_deg=0.0, timeout=10):
    """
    Query TomTom Routing by creating a micro-route from (lat,lon) to a nearby offset point.
    Returns integer lane count or None.
    """
    lat2, lon2 = offset_point(lat, lon, meters_forward=meters_ahead, bearing_deg=bearing_deg)
    url = TOMTOM_ROUTE_URL.format(start=f"{lat:.6f},{lon:.6f}", end=f"{lat2:.6f},{lon2:.6f}")
    params = {
        "key": TOMTOM_API_KEY,
        "instructionsType": "tagged",
        "sectionType": "lanes",
        "traffic": "false",
    }
    headers = {"User-Agent": "LaneCountScript-TomTom/1.0"}

    try:
        r = requests.get(url, params=params, headers=headers, timeout=timeout)
        if r.status_code != 200:
            return None
        data = r.json()
    except Exception:
        return None

    return extract_lane_count_from_tomtom(data)

# --- main CSV processing -----------------------------------------------------

def add_lane_counts_to_csv(input_csv_path, output_csv_path, sleep_between=0.5, max_retries=3):
    """
    Reads input CSV, fetches lane counts, writes output with lane_count column.
    """
    with open(input_csv_path, newline='') as infile, open(output_csv_path, 'w', newline='') as outfile:
        reader = csv.DictReader(infile)
        fieldnames = (reader.fieldnames or []) + ["lane_count"]
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()

        total = found = 0
        for row in reader:
            total += 1
            lat = row.get("lat") or row.get("latitude")
            lon = row.get("lon") or row.get("longitude")

            lane_count = None
            if lat is not None and lon is not None:
                try:
                    lat_f, lon_f = float(lat), float(lon)
                    # Retry loop
                    for attempt in range(max_retries):
                        lane_count = get_lane_count_tomtom(lat_f, lon_f)
                        if lane_count is not None:
                            break
                        time.sleep(0.5 * (attempt + 1))
                except ValueError:
                    pass

            if lane_count is None:
                row["lane_count"] = "N/A"
            else:
                row["lane_count"] = str(lane_count)
                found += 1

            writer.writerow(row)
            time.sleep(sleep_between)

    print(f"[done] Processed {total} locations, found lane data for {found}.")
    print(f"Output saved to {output_csv_path}")

In [ ]:
input_path =r"C:\Users\HP\Documents\base\sample_lat_lon.csv"
output_path = r"C:\Users\HP\Documents\base\sample_lat_lon_tomtom_result23.csv"

add_lane_counts_to_csv(input_path, output_path)